# 02 — Features: das três ômicas às matrizes amostra × gene

Notebook equivalente a `src/features/`. Cobre as etapas 1 a 4 de `.claude/specs/etapas_do_projeto.md`:

1. **Resumo por gene**: uma matriz amostra × gene por ômica.
2. **Universo comum**: mesmos genes e mesmas amostras nas três.
3. **Ordenação cromossômica (hg19)** e tabela gene → posição.
4. **Matrizes salvas** em `data/interim/`, com valores brutos e ausente como `NaN`, nunca 0.

Entrada: `data/raw/`, preenchido pelo `01_data`. Decisões aplicadas (2026-10-08, ver diário):

| Canal | Resumo por gene |
|---|---|
| Mutação | contagem de variantes não silenciosas (9 classes do maftools; sem Splice_Region), símbolos do MAF levados aos do RefGene via HGNC |
| CNV | log2 dos segmentos sobre o gene, média ponderada pelo trecho sobreposto |
| Metilação | média do beta no promotor Illumina estrito (TSS1500 + TSS200), por coordenada com o RefGene; **X anulado** (sexo) |

Genes: transcritos codificantes (`NM_`) do RefGene hg19, sem símbolos com posição ambígua. Y fora; X
dentro, mas sem metilação.

Saída: `data/interim/{mutacao,cnv,metilacao}.parquet` (transpostas, gene × amostra), `genes.csv`,
`amostras.csv`.

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
RAW, INTERIM = RAIZ / "data" / "raw", RAIZ / "data" / "interim"

CROMOSSOMOS = [str(i) for i in range(1, 23)] + ["X", "Y"]
CLASSES = ["STAD_CIN", "STAD_MSI", "STAD_GS", "STAD_EBV"]
NAO_SILENCIOSAS = ["Missense_Mutation", "Nonsense_Mutation", "Nonstop_Mutation", "Frame_Shift_Del", "Frame_Shift_Ins",
                   "In_Frame_Del", "In_Frame_Ins", "Splice_Site", "Translation_Start_Site"]
JANELA_PROMOTOR = 1500  # TSS1500 + TSS200 da Illumina = até 1.500 pb a montante do TSS


def normalizar_cromossomos(serie: pd.Series) -> pd.Series:
    c = serie.astype(str).str.removeprefix("chr").replace({"23": "X", "24": "Y"})
    invalidos = sorted(set(c) - set(CROMOSSOMOS))
    if invalidos:
        raise ValueError(f"cromossomos fora de 1–22/X/Y: {invalidos}")
    return c


ler_json = lambda nome: json.loads((RAW / nome).read_text())  # noqa: E731
rotulo = pd.DataFrame(ler_json("subtipos.json")).set_index("patientId")["value"]
amostras = {omica: sorted(ler_json(f"amostras_{lista}.json"))
            for omica, lista in {"mutação": "sequenced", "CNV": "log2CNA", "metilação": "methylation_hm450"}.items()}
print({o: len(a) for o, a in amostras.items()})

{'mutação': 436, 'CNV': 438, 'metilação': 393}


## 1. Genes — RefGene hg19

Só transcritos codificantes (`NM_`), coerente com um MAF de exoma e com a contagem de não silenciosas.
O Y sai aqui (decisão 5). O TSS respeita a fita: na fita `-`, o gene começa no `txEnd`.

In [2]:
COLUNAS_REFGENE = ["bin", "name", "chrom", "strand", "txStart", "txEnd", "cdsStart", "cdsEnd", "exonCount",
                   "exonStarts", "exonEnds", "score", "name2", "cdsStartStat", "cdsEndStat", "exonFrames"]
LIMITE_EXTENSAO = 2.5e6  # maior gene humano tem ~2,3 Mb; acima disso é o mesmo símbolo em loci distantes (parálogos)

tx = pd.read_csv(RAW / "refGene_hg19.txt.gz", sep="\t", header=None, names=COLUNAS_REFGENE,
                 usecols=["name", "chrom", "strand", "txStart", "txEnd", "name2"])
n_total = len(tx)
tx = tx[tx.name.str.startswith("NM_") & tx.chrom.str.removeprefix("chr").isin(CROMOSSOMOS[:-1])].copy()
tx["chrom"] = normalizar_cromossomos(tx.chrom)
tx["tss"] = tx.txStart.where(tx.strand == "+", tx.txEnd)
print(f"transcritos: {n_total:,} → {len(tx):,} codificantes em 1–22/X")

# mesmo símbolo em mais de um cromossomo: sem posição única, não dá para ordenar — sai, contado
n_chrom = tx.groupby("name2").chrom.nunique()
print(f"genes em mais de um cromossomo (excluídos): {(n_chrom > 1).sum()} {n_chrom[n_chrom > 1].index[:8].tolist()}")
tx = tx[~tx.name2.isin(n_chrom.index[n_chrom > 1])]

genes = tx.groupby("name2").agg(cromossomo=("chrom", "first"), inicio=("txStart", "min"), fim=("txEnd", "max"),
                                fita=("strand", "first"), n_transcritos=("name", "size")).rename_axis("gene")
genes["extensao"] = genes.fim - genes.inicio
print(genes.sort_values("extensao", ascending=False).head(13)[["cromossomo", "inicio", "fim", "n_transcritos", "extensao"]])

# mesmo símbolo em loci distantes do mesmo cromossomo: posição e CNV sem sentido (média sobre dezenas de Mb) — sai
longos = genes.index[genes.extensao > LIMITE_EXTENSAO]
genes, tx = genes.drop(longos), tx[~tx.name2.isin(longos)]
print(f"\nexcluídos por extensão > 2,5 Mb: {len(longos)} · restam {len(genes):,} genes · "
      f"extensão mediana {genes.extensao.median() / 1e3:.0f} kb, máxima {genes.extensao.max() / 1e6:.2f} Mb")

transcritos: 81,407 → 59,343 codificantes em 1–22/X
genes em mais de um cromossomo (excluídos): 4 ['OR4F16', 'OR4F29', 'OR4F3', 'RPL21']
          cromossomo     inicio        fim  n_transcritos  extensao
gene                                                               
FAM72C             1  143896524  206155074              6  62258550
FOXD4L4            9   42718065   70428899              2  27710834
NOTCH2NLC          1  120391203  145283180              2  24891977
SPATA31A5          9   41321106   65509610              3  24188504
SPATA31A7          9   41321109   65509610              3  24188501
PPIAL4C            1  144363461  149553787              3   5190326
PPIAL4A            1  144363586  149553662              3   5190076
FAM25E            10   46310875   51216406              2   4905531
NBPF19             1  144146811  148346791              2   4199980
PPIAL4H            1  143765472  147955536              2   4190064
RGPD5              2  110550334  113191191     

## 2. Mutação — contagem de variantes não silenciosas por gene

Amostra sequenciada sem variante no gene = **0**.

O MAF usa símbolos HGNC atuais; o RefGene (2020), às vezes, o nome anterior do mesmo gene. Quando o
símbolo não casa direto, a ponte é o Entrez ID do MAF no HGNC:
1. tentam-se os símbolos anteriores (`prev_symbol`) e depois os sinônimos (`alias_symbol`);
2. só se aceita candidato **único**;
3. o candidato não pode ser outro gene do MAF.

Gene sem par continua fora, contado.

In [3]:
mut = pd.json_normalize(ler_json("mutacoes_api.json"))
mut = mut.rename(columns={"gene.hugoGeneSymbol": "gene", "sampleId": "amostra", "mutationType": "classe"})
n_mut = len(mut)
mut = mut[mut.classe.isin(NAO_SILENCIOSAS)].copy()
print(f"variantes: {n_mut:,} → não silenciosas {len(mut):,}")

# a API devolve símbolos em maiúsculas ('C6ORF118'); HGNC e RefGene usam 'C6orf118'
por_maiuscula = pd.Series(genes.index, index=genes.index.str.upper())
assert por_maiuscula.index.is_unique, "símbolos do RefGene que só diferem por maiúsculas"
mut["gene"] = mut.gene.where(mut.gene.isin(genes.index), mut.gene.map(por_maiuscula).fillna(mut.gene))

hgnc = (pd.read_csv(RAW / "hgnc_complete_set.txt", sep="\t", low_memory=False,
                    usecols=["symbol", "alias_symbol", "prev_symbol", "entrez_id"])
        .dropna(subset=["entrez_id"]).drop_duplicates("entrez_id"))
hgnc = hgnc.set_index(hgnc.entrez_id.astype(int))
diretos = set(mut.gene) & set(genes.index)


def ponte(entrez: int) -> str | None:
    if entrez not in hgnc.index:
        return None
    for campo in ["prev_symbol", "alias_symbol"]:
        candidatos = {c for c in str(hgnc.at[entrez, campo]).split("|") if c in genes.index and c not in diretos}
        if len(candidatos) == 1:
            return candidatos.pop()
    return None


sem_par = mut.loc[~mut.gene.isin(genes.index), ["gene", "entrezGeneId"]].drop_duplicates("gene")
recuperados = {g: r for g, e in zip(sem_par.gene, sem_par.entrezGeneId) if (r := ponte(int(e)))}
mut["gene_refgene"] = mut.gene.where(mut.gene.isin(genes.index), mut.gene.map(recuperados))
fora = mut.gene_refgene.isna()
print(f"sem par direto: {len(sem_par):,} genes · recuperados pelo HGNC: {len(recuperados):,} "
      f"(ex.: {dict(list(recuperados.items())[:6])}) · alvos repetidos: {pd.Series(recuperados).duplicated().sum()}")
print(f"ainda fora: {fora.sum():,} variantes ({fora.mean():.1%}) em {mut.gene[fora].nunique():,} genes; "
      f"mais mutados: {mut.gene[fora].value_counts().head(8).to_dict()}")

mutacao = (mut[~fora].groupby(["amostra", "gene_refgene"]).size().unstack(fill_value=0)
           .reindex(index=amostras["mutação"], columns=genes.index, fill_value=0).astype("int16"))
print(f"matriz {mutacao.shape} · amostras sem nenhuma variante: {(mutacao.sum(axis=1) == 0).sum()} · "
      f"genes nunca mutados: {(mutacao.sum() == 0).sum():,}")

variantes: 147,953 → não silenciosas 146,113


sem par direto: 578 genes · recuperados pelo HGNC: 231 (ex.: {'SLC9D1': 'TMCO3', 'CLXN': 'EFCAB1', 'QTMAN': 'GTDC1', 'SKIC3': 'TTC37', 'HEMK2': 'N6AMT1', 'FHIP2B': 'FAM160B2'}) · alvos repetidos: 0
ainda fora: 939 variantes (0.6%) em 347 genes; mais mutados: {'CRIPAK': 19, 'IGHM': 17, 'IGHG1': 14, 'AGAP7P': 13, 'UBBP4': 13, 'RTEL1-TNFRSF6B': 12, 'IGHV1OR21-1': 12, 'IGHA1': 11}


matriz (436, 19327) · amostras sem nenhuma variante: 1 · genes nunca mutados: 2,290


## 3. CNV — log2 dos segmentos sobre o gene

Para cada gene, é a média dos segmentos que o cobrem, ponderada pelo trecho sobreposto. Gene sem
segmento = `NaN`.

In [4]:
def media_por_sobreposicao(g_ini, g_fim, s_ini, s_fim, valor):
    sobre = np.clip(np.minimum(g_fim[:, None], s_fim[None, :]) - np.maximum(g_ini[:, None], s_ini[None, :]), 0, None)
    peso = sobre.sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(peso > 0, (sobre @ valor) / peso, np.nan)


# autoteste: gene dentro de 1 segmento; gene dividido meio a meio; gene fora de qualquer segmento
r = media_por_sobreposicao(np.array([0, 50, 300]), np.array([100, 150, 400]),
                           np.array([0, 100]), np.array([100, 200]), np.array([1.0, 3.0]))
assert np.allclose(r[:2], [1.0, 2.0]) and np.isnan(r[2])
print("media_por_sobreposicao: ok")

media_por_sobreposicao: ok


In [5]:
seg = pd.read_csv(RAW / "data_cna_hg19.seg", sep="\t")
seg.columns = ["amostra", "chrom", "inicio", "fim", "n_marcadores", "log2"]
seg["chrom"] = normalizar_cromossomos(seg.chrom)

linha = {a: i for i, a in enumerate(amostras["CNV"])}
M = np.full((len(amostras["CNV"]), len(genes)), np.nan, dtype=np.float32)
for chrom, g in genes.groupby("cromossomo"):
    colunas = genes.index.get_indexer(g.index)
    gi, gf = g.inicio.to_numpy(), g.fim.to_numpy()
    for amostra, s in seg[seg.chrom == chrom].groupby("amostra"):
        M[linha[amostra], colunas] = media_por_sobreposicao(gi, gf, s.inicio.to_numpy(), s.fim.to_numpy(), s.log2.to_numpy())
cnv = pd.DataFrame(M, index=amostras["CNV"], columns=genes.index)
print(f"matriz {cnv.shape} · células NaN: {cnv.isna().mean().mean():.2%} · log2 de {np.nanmin(M):.2f} a {np.nanmax(M):.2f}")

# genes que nenhum segmento cobre, em nenhuma amostra: onde estão?
sem_cnv = genes.loc[cnv.columns[cnv.isna().all()]]
print(f"genes sem CNV em nenhuma amostra: {len(sem_cnv)} · por cromossomo: {sem_cnv.cromossomo.value_counts().head(6).to_dict()} · "
      f"extensão mediana {sem_cnv.extensao.median() / 1e3:.0f} kb (todos: {genes.extensao.median() / 1e3:.0f} kb)")
sem_cnv.sort_values(["cromossomo", "inicio"]).head(8)

matriz (438, 19327) · células NaN: 2.78% · log2 de -5.54 a 4.43
genes sem CNV em nenhuma amostra: 510 · por cromossomo: {'1': 109, '16': 45, '22': 41, '20': 37, '8': 37, '4': 24} · extensão mediana 14 kb (todos: 25 kb)


,cromossomo,inicio,fim,fita,n_transcritos,extensao
gene,,,,,,
OR4F5,1,69090,70008,+,1,918
SAMD11,1,859302,879954,+,3,20652
NOC2L,1,879582,894636,-,1,15054
KLHL17,1,895963,901099,+,1,5136
PLEKHN1,1,901861,911245,+,3,9384
PERM1,1,910577,917473,-,4,6896
HES4,1,934343,935477,-,2,1134
ISG15,1,948876,949919,+,1,1043


## 4. Metilação — beta médio no promotor (Illumina estrito)

Uma sonda é do promotor de um transcrito se está até 1.500 pb a montante do TSS, que é a união de
TSS1500 e TSS200. O promotor do gene é a união dos promotores dos seus transcritos. A coordenada das
sondas vem do probeMap do Xena (hg19).

In [6]:
mapa = pd.read_csv(RAW / "probemap_450_hg19.tsv", sep="\t").rename(columns={"#id": "sonda"}).dropna(subset=["chrom"])
mapa["chrom"] = normalizar_cromossomos(mapa.chrom)

pares = []
for chrom, t in tx.groupby("chrom"):
    p = mapa[mapa.chrom == chrom].sort_values("chromStart")
    pos = p.chromStart.to_numpy()
    mais = t.strand.to_numpy() == "+"
    tss = t.tss.to_numpy()
    lo = np.where(mais, tss - JANELA_PROMOTOR, tss)   # fita +: [TSS − 1500, TSS) · fita −: [TSS, TSS + 1500)
    hi = np.where(mais, tss, tss + JANELA_PROMOTOR)
    a, b = np.searchsorted(pos, lo), np.searchsorted(pos, hi)
    idx = np.concatenate([np.arange(x, y) for x, y in zip(a, b)])
    pares.append(pd.DataFrame({"sonda": p.sonda.to_numpy()[idx], "gene": t.name2.to_numpy()[np.repeat(np.arange(len(t)), b - a)]}))
promotor = pd.concat(pares).drop_duplicates()
por_gene = promotor.groupby("gene").size()
print(f"pares sonda–gene: {len(promotor):,} · sondas: {promotor.sonda.nunique():,} · genes com promotor: {len(por_gene):,} · "
      f"mediana de sondas por gene: {por_gene.median():.0f} · sondas em mais de um gene: {(promotor.sonda.value_counts() > 1).sum():,}")

pares sonda–gene: 123,845 · sondas: 116,628 · genes com promotor: 18,141 · mediana de sondas por gene: 6 · sondas em mais de um gene: 7,057


Leitura do arquivo de 1,44 GB em blocos: guarda a anotação de todas as sondas, para a validação, e o
beta só das sondas de promotor.

In [7]:
caminho = RAW / "data_methylation_hm450.txt"
with open(caminho) as f:
    pular = next(i for i, linha in enumerate(f) if not linha.startswith("#"))
sondas_promotor = set(promotor.sonda)
anotacao, betas = [], []
for bloco in pd.read_csv(caminho, sep="\t", skiprows=pular, chunksize=25_000):
    bloco = bloco.set_index("ENTITY_STABLE_ID")
    anotacao.append(bloco[["NAME", "DESCRIPTION"]])
    betas.append(bloco.loc[bloco.index.isin(sondas_promotor), amostras["metilação"]].astype("float32"))
anotacao, betas = pd.concat(anotacao), pd.concat(betas)
print(f"sondas no arquivo: {len(anotacao):,} · de promotor lidas: {len(betas):,} de {len(sondas_promotor):,} · "
      f"beta NaN: {betas.isna().mean().mean():.2%} · faixa {np.nanmin(betas.values):.3f}–{np.nanmax(betas.values):.3f}")

sondas no arquivo: 396,065 · de promotor lidas: 116,628 de 116,628 · beta NaN: 0.09% · faixa 0.004–0.995


### Validação contra a anotação Illumina do arquivo

O gabarito são as sondas de **um gene só** no arquivo, cujo símbolo existe no RefGene. Pela Illumina,
elas são de promotor se a região inclui TSS1500 ou TSS200. A conta por coordenada deve concordar com
isso.

In [8]:
unico = anotacao[anotacao.NAME.notna() & ~anotacao.NAME.str.contains(";", na=False) & anotacao.NAME.isin(genes.index)]
gabarito = unico.DESCRIPTION.str.contains("TSS1500|TSS200", na=False)
pares_promotor = pd.MultiIndex.from_arrays([promotor.sonda, promotor.gene])
calculado = pd.Series(pd.MultiIndex.from_arrays([unico.index, unico.NAME]).isin(pares_promotor), index=unico.index)
tabela = pd.crosstab(gabarito.rename("Illumina: promotor"), calculado.rename("coordenada: promotor"))
print(f"sondas de 1 gene usadas: {len(unico):,} · concordância {(gabarito == calculado).mean():.1%} · "
      f"sensibilidade {(calculado & gabarito).sum() / gabarito.sum():.1%} · "
      f"precisão {(calculado & gabarito).sum() / calculado.sum():.1%}")
tabela

sondas de 1 gene usadas: 244,915 · concordância 95.0% · sensibilidade 93.5% · precisão 91.6%


coordenada: promotor,False,True
Illumina: promotor,,
False,156717,6956
True,5287,75955


In [9]:
# beta do gene = média das sondas do seu promotor que têm valor naquela amostra
p = promotor[promotor.sonda.isin(betas.index)]
linhas_g, colunas_s = pd.factorize(p.gene)[0], betas.index.get_indexer(p.sonda)
genes_met = pd.factorize(p.gene)[1]
incid = sparse.csr_matrix((np.ones(len(p), dtype=np.float32), (linhas_g, colunas_s)), shape=(len(genes_met), len(betas)))
valores = betas.to_numpy()
soma, n = incid @ np.nan_to_num(valores), incid @ (~np.isnan(valores)).astype(np.float32)
with np.errstate(invalid="ignore", divide="ignore"):
    metilacao = pd.DataFrame((soma / n).T, index=betas.columns, columns=genes_met).astype("float32")
print(f"matriz {metilacao.shape} · células NaN: {metilacao.isna().mean().mean():.2%}")

classe = rotulo.reindex(metilacao.index.str[:12]).values
pd.DataFrame({g: metilacao[g].groupby(classe).median() for g in ["MLH1", "CDKN2A"]}).reindex(CLASSES).round(3)

matriz (393, 18141) · células NaN: 0.02%


,MLH1,CDKN2A
STAD_CIN,0.086,0.169
STAD_MSI,0.525,0.241
STAD_GS,0.115,0.189
STAD_EBV,0.125,0.542


### Efeito do sexo na metilação do X (decisão 5)

Nas mulheres, um dos X é inativado por metilação. Se a metilação dos promotores do X separar os sexos,
o canal carrega o sexo, e o sexo está associado ao subtipo. Medidas:
- **Δ mediana** (mulheres − homens) por gene, no X e nos autossomos (referência);
- **AUC** de um escore único (beta médio dos promotores do X) para separar os sexos. 0,5 = não separa;
  1,0 = separa perfeitamente.

In [10]:
from scipy.stats import mannwhitneyu

sexo = pd.DataFrame(ler_json("sexo.json")).set_index("patientId")["value"]
sexo_amostra = sexo.reindex(metilacao.index.str[:12]).to_numpy()
mulher, homem = sexo_amostra == "Female", sexo_amostra == "Male"
no_x = genes.cromossomo.reindex(metilacao.columns).eq("X").to_numpy()
print(f"amostras com metilação: {mulher.sum()} mulheres, {homem.sum()} homens · genes do X com promotor: {no_x.sum():,}")

delta = metilacao[mulher].median() - metilacao[homem].median()
print(pd.DataFrame({regiao: {"genes": len(d), "|Δ| mediano": round(d.abs().median(), 3),
                             "% com |Δ| > 0,2": round(100 * (d.abs() > 0.2).mean(), 1)}
                    for regiao, d in {"X": delta[no_x], "autossomos": delta[~no_x]}.items()}).T)

escore_x = metilacao.loc[:, no_x].mean(axis=1)
u = mannwhitneyu(escore_x[mulher], escore_x[homem]).statistic
print(f"AUC do escore do X para separar os sexos: {u / (mulher.sum() * homem.sum()):.3f}")

subtipo = rotulo.reindex(metilacao.index.str[:12]).to_numpy()
pd.crosstab(pd.Series(subtipo, name="subtipo"), pd.Series(sexo_amostra, name="sexo"), normalize="index").reindex(CLASSES).round(2)

amostras com metilação: 135 mulheres, 258 homens · genes do X com promotor: 713


              genes  |Δ| mediano  % com |Δ| > 0,2
X             713.0        0.118             39.3
autossomos  17428.0        0.003              0.0
AUC do escore do X para separar os sexos: 0.929


sexo,Female,Male
subtipo,,
STAD_CIN,0.32,0.68
STAD_MSI,0.47,0.53
STAD_GS,0.30,0.70
STAD_EBV,0.10,0.90


**Resultado: o efeito é forte.** O escore do X separa os sexos com AUC acima de 0,9, e cerca de 40% dos
genes do X têm diferença de beta > 0,2 entre os sexos, contra ~0% nos autossomos. Como o sexo está
associado ao subtipo (EBV ~90% homens), o canal de metilação do X entregaria o sexo ao modelo como atalho.

**Decisão 5 aplicada:** o X sai **só do canal de metilação**. Os genes do X continuam no universo, com
mutação e CNV, e a metilação deles fica `NaN`, ausente por decisão e não por falta de dado. É uma exceção
explícita à regra "presente nas três ômicas", tratada no universo comum abaixo.

In [11]:
genes_x = set(genes.index[genes.cromossomo == "X"])
no_x_met = metilacao.columns.isin(list(genes_x))
metilacao.loc[:, no_x_met] = np.nan
print(f"metilação anulada em {no_x_met.sum()} genes do X · genes do X na tabela: {len(genes_x)}")

metilação anulada em 713 genes do X · genes do X na tabela: 848


## 5. Universo comum

Amostras com as três ômicas. Gene "presente" numa ômica = tem valor (não `NaN`) em pelo menos uma dessas
amostras. Na mutação, todo gene codificante está presente: amostra sequenciada sem variante é 0.
**Exceção (decisão 5):** os genes do X contam como presentes na metilação, que fica toda `NaN` para eles.

In [12]:
amostras_comuns = sorted(set(mutacao.index) & set(cnv.index) & set(metilacao.index))
presentes = {"mutação": set(mutacao.columns),
             "CNV": set(cnv.columns[cnv.loc[amostras_comuns].notna().any()]),
             "metilação": set(metilacao.columns[metilacao.loc[amostras_comuns].notna().any()]) | genes_x}
genes_comuns = set.intersection(*presentes.values())
print("genes presentes:", {o: len(g) for o, g in presentes.items()}, f"→ universo: {len(genes_comuns):,}")
print("autossômicos perdidos por falta de sonda de promotor, por cromossomo:",
      genes.loc[list(presentes["CNV"] - genes_comuns), "cromossomo"].value_counts().head(5).to_dict())

perda = pd.DataFrame({o: rotulo.reindex([a[:12] for a in m.index]).value_counts()
                      for o, m in {"mutação": mutacao, "CNV": cnv, "metilação": metilacao}.items()})
perda["3 ômicas"] = rotulo.reindex([a[:12] for a in amostras_comuns]).value_counts()
perda.reindex(CLASSES + ["STAD_POLE"]).fillna(0).astype(int).rename_axis("classe")

genes presentes: {'mutação': 19327, 'CNV': 18817, 'metilação': 18276} → universo: 17,818
autossômicos perdidos por falta de sonda de promotor, por cromossomo: {'1': 120, '11': 105, '9': 105, '7': 79, '19': 64}


,mutação,CNV,metilação,3 ômicas
classe,,,,
STAD_CIN,223,223,202,202
STAD_MSI,73,73,59,59
STAD_GS,50,50,46,46
STAD_EBV,30,30,29,29
STAD_POLE,7,7,7,7


## 6. Ordenação cromossômica

As colunas seguem a ordem cromossomo (1…22, X) → início do gene, e é a mesma ordem nos três canais.
A tabela gene → posição vai junto: é ela que leva um pixel de volta ao genoma.

In [13]:
ordem_crom = {c: i for i, c in enumerate(CROMOSSOMOS)}
tabela_genes = (genes.loc[sorted(genes_comuns)].assign(_c=lambda d: d.cromossomo.map(ordem_crom))
                .sort_values(["_c", "inicio"]).drop(columns="_c"))
tabela_genes["posicao"] = np.arange(len(tabela_genes))
print(tabela_genes.cromossomo.value_counts().reindex(CROMOSSOMOS).dropna().astype(int).to_dict())
tabela_genes.head()

{'1': 1771, '2': 1164, '3': 998, '4': 660, '5': 799, '6': 966, '7': 813, '8': 598, '9': 643, '10': 680, '11': 1151, '12': 932, '13': 306, '14': 554, '15': 529, '16': 733, '17': 1089, '18': 243, '19': 1314, '20': 475, '21': 200, '22': 375, 'X': 825}


,cromossomo,inicio,fim,fita,n_transcritos,extensao,posicao
gene,,,,,,,
PRDM16,1,2985766,3355185,+,2,369419,0
ARHGEF16,1,3371228,3397677,+,1,26449,1
MEGF6,1,3404514,3528072,-,1,123558,2
TPRG1L,1,3541578,3546691,+,1,5113,3
WRAP73,1,3547333,3566667,-,1,19334,4


## 7. Matrizes salvas — valores brutos

Em `data/interim/`. As matrizes são gravadas **transpostas** (gene nas linhas): o parquet é colunar, e
~20 mil colunas custam mais em metadado que em dado (lição do projeto irmão). Para ler, use
`pd.read_parquet(...).T`.

In [14]:
INTERIM.mkdir(parents=True, exist_ok=True)
ordem = tabela_genes.index
# reindex: genes do X sem sonda de promotor entram na metilação como colunas NaN (decisão 5)
matrizes = {nome: m.reindex(index=amostras_comuns, columns=ordem)
            for nome, m in {"mutacao": mutacao, "cnv": cnv, "metilacao": metilacao}.items()}
for nome, m in matrizes.items():
    m.T.to_parquet(INTERIM / f"{nome}.parquet")
tabela_genes.to_csv(INTERIM / "genes.csv")
tabela_amostras = pd.DataFrame({"amostra": amostras_comuns, "paciente": [a[:12] for a in amostras_comuns]})
tabela_amostras["subtipo"] = tabela_amostras.paciente.map(rotulo)
tabela_amostras.to_csv(INTERIM / "amostras.csv", index=False)
for f in sorted(INTERIM.iterdir()):
    print(f"{f.name}: {f.stat().st_size / 1e6:.1f} MB")

amostras.csv: 0.0 MB
cnv.parquet: 1.0 MB
genes.csv: 0.8 MB
metilacao.parquet: 39.0 MB
mutacao.parquet: 0.7 MB


## 8. Conferências (checklist de `etapas_do_projeto.md`)

Relê do disco e confere o que a spec pede.

In [15]:
lidas = {nome: pd.read_parquet(INTERIM / f"{nome}.parquet").T for nome in matrizes}
genes_lidos = pd.read_csv(INTERIM / "genes.csv", index_col="gene")

# universo comum: mesmas linhas e colunas, na mesma ordem, nas três
for nome, m in lidas.items():
    assert list(m.index) == amostras_comuns, f"{nome}: amostras fora do universo/ordem"
    assert list(m.columns) == list(genes_lidos.index), f"{nome}: genes fora da ordem da tabela"
# ordenação cromossômica: posição crescente e cromossomos em bloco, na ordem 1…22, X
assert (genes_lidos.posicao.diff().dropna() == 1).all()
assert genes_lidos.cromossomo.map(ordem_crom).is_monotonic_increasing
# valores brutos e NaN preservado (o NaN de antes de salvar é o NaN de depois)
for nome, m in lidas.items():
    assert m.isna().sum().sum() == matrizes[nome].isna().sum().sum(), f"{nome}: NaN mudou ao salvar"
assert lidas["mutacao"].isna().sum().sum() == 0 and (lidas["mutacao"] >= 0).all().all()
assert 0 <= np.nanmin(lidas["metilacao"].values) and np.nanmax(lidas["metilacao"].values) <= 1
# decisão 5: metilação do X toda ausente; todo gene autossômico tem metilação
x = (genes_lidos.cromossomo == "X").to_numpy()
assert lidas["metilacao"].loc[:, x].isna().all().all(), "metilação do X deveria ser toda NaN"
assert lidas["metilacao"].loc[:, ~x].notna().any().all(), "gene autossômico sem metilação no universo"

pd.DataFrame({nome: {"amostras": m.shape[0], "genes": m.shape[1], "NaN %": round(100 * m.isna().mean().mean(), 2),
                     "mín": float(np.nanmin(m.values)), "mediana": float(np.nanmedian(m.values)), "máx": float(np.nanmax(m.values))}
              for nome, m in lidas.items()})

,mutacao,cnv,metilacao
amostras,389.0,389.0000,389.000000
genes,17818.0,17818.0000,17818.000000
NaN %,0.0,0.1100,4.650000
mín,0.0,-4.7424,0.006313
mediana,0.0,-0.0067,0.177833
máx,24.0,4.4332,0.991732
